<a href="https://colab.research.google.com/github/Himkeshtak/AgenticAI-GFG/blob/main/RAG_Implementation_basic_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install faiss-cpu
!pip install langchain==0.1.16

from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.4
    Uninstalling tenacity-9.1.4:
      Successfully uninstalled tenacity-9.1.4
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Succe

In [1]:
import faiss
import numpy as np
import re

index = faiss.IndexFlatL2(768)

np.random.seed(42)
context_data = np.random.random((100, 768)).astype('float32')

index.add(context_data)

print(f"Indexed {index.ntotal} context vectors.")

Indexed 100 context vectors.


In [2]:
def semantic_search(query_embedding, index, top_k=5):
    distances, indices = index.search(query_embedding, top_k)
    return indices


In [3]:
query_embedding = np.random.random((1, 768)).astype(
    'float32')
retrieved_indices = semantic_search(query_embedding, index)
print(f"Retrieved document indices: {retrieved_indices}")

Retrieved document indices: [[26 38 11 78 12]]


In [4]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
from langchain_core.prompts import PromptTemplate
context_texts = [
    "Retrieval-Augmented Generation combines a retriever and generator.",
    "It reduces hallucinations by grounding answers in retrieved documents.",
    "Uses Dense Passage Retrieval for semantic search.",
    "Employs Fusion-in-Decoder and Fusion-in-Encoder techniques.",
    "Provides up-to-date and domain-specific responses."
]

prompt_template = PromptTemplate(
    input_variables=["chat_history", "question", "context_texts"],
    template="Chat history: {chat_history}\nQuestion: {question}\nContext: {context_texts}\nAnswer:"
)

Initialize Memory and Build Chat Function
We will initialize memory to our system:

memory_key: identifies where conversation history is stored.

return_messages=False:returns text as plain string instead of structured messages.

Memory enables the agent to remember previous questions/answers.

Retrieve documents: semantic_search finds relevant context.

Load conversation: includes past Q&A from memory.

Format prompt: combines context, history and user query.

Generate response: GPT-2 predicts answer.

Post-process: clean up newlines and spaces.

Update memory: conversation history is saved for future queries.

In [8]:
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(
    memory_key="chat_history", return_messages=False)


def chat(question):
    query_embedding = np.random.rand(1, 768).astype("float32")
    retrieved_indices = semantic_search(query_embedding, index)
    context_texts_for_prompt = [f"Document {i}" for i in retrieved_indices[0]]

    chat_history = memory.load_memory_variables({}).get("chat_history", "")

    prompt = prompt_template.format(
        chat_history=chat_history,
        question=question,
        context_texts="\n".join(context_texts_for_prompt)
    )

    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        inputs,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response[len(prompt):].strip()
    response = re.sub(r"[\r\n]+", " ", response)

    memory.chat_memory.add_user_message(question)
    memory.chat_memory.add_ai_message(response)

    return response

In [9]:
print(chat("What is Retrieval-Augmented Generation (RAG)?"))
print(chat("Explain the role of memory in this system."))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Retrieval-Augmented Generation is a new feature in the RAG specification. It is a new feature that allows you to create a new document with a Retrieval-Augmented Generation (RAG) document. Question: What is Retrieval-Augmented Generation (RAG)? Context: Document 37 Document 75 Document 72 Document 65 Document 68 Answer: Retrieval-Augmented Generation is a new feature in the RAG specification. It is a new
Retrieval-Augmented Generation is a new feature in the RAG specification. It is a new feature that allows you to create a new document with a Retrieval-Augmented Generation (RAG) document. Question: What is Retrieval-Augmented Generation (RAG)? Context: Document 7 Document 55 Document 55 Document 55 Document 55 Answer: Retrieval-Augmented Generation is a new feature in the RAG specification. It is a new feature that allows you to create a
